# 02 - Leakage-safe split (Day 3)
**Decision (Q6):** split MVTec `test/` 50/50, stratified by `category/defect_type`, fixed seed 42.

| Role | What's in it | Used by |
|---|---|---|
| `train_good` | all 3629 `train/` images (all good) | PatchCore memory bank; VLM may sample good examples |
| `vlm_train` | half of `test/` (good + defect) | VLM fine-tuning only |
| `eval` | other half of `test/` | **both** models are scored here; neither trains on it |

**Known weaknesses (say these in interviews):**
1. Eval is half the published test set, so our PatchCore number will not match the paper exactly; we re-run PatchCore on *our* eval half.
2. Rare types (e.g. wood/color, 8 images) get ~4 eval images, so per-type scores are noisy. Report per category and per kind (object/texture), not per rare type.


In [ ]:
import os
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', 200)

candidates = [p.parent for p in Path('/kaggle/input').rglob('bottle') if p.is_dir() and (p / 'train').exists()]
assert candidates, 'MVTec AD not found - attach it via Add Input'
ROOT = candidates[0]
print('ROOT =', ROOT)

## Step 1 - Rebuild the manifest (same code as Day 2)

In [ ]:
TEXTURES = {'carpet', 'grid', 'leather', 'tile', 'wood'}
IMG_EXT = {'.png', '.jpg', '.jpeg'}
rows = []
for cat_dir in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    cat = cat_dir.name
    for split in ['train', 'test']:
        split_dir = cat_dir / split
        if not split_dir.exists():
            continue
        for dtype_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            dtype = dtype_dir.name
            for img in sorted(dtype_dir.iterdir()):
                if img.suffix.lower() not in IMG_EXT:
                    continue
                mask = cat_dir / 'ground_truth' / dtype / f'{img.stem}_mask.png'
                rows.append(dict(
                    category=cat,
                    kind='texture' if cat in TEXTURES else 'object',
                    split=split,
                    defect_type=dtype,
                    label='good' if dtype == 'good' else 'defect',
                    path=str(img),
                    mask_path=str(mask) if dtype != 'good' else None,
                ))
manifest = pd.DataFrame(rows)
assert len(manifest) == 5354, f'expected 5354 rows, got {len(manifest)}'
print(len(manifest), 'images ✓')

## Step 2 - The split function
Identical to `src/defect_vlm/split.py` in the repo (where it is unit-tested).

In [ ]:
"""Leakage-safe split of MVTec AD for VLM fine-tuning vs evaluation.

Roles assigned to every image:
  train_good : MVTec train/ images (all good). PatchCore memory bank; the VLM
               may also sample "good" examples from here.
  vlm_train  : half of MVTec test/ (good + defect). The VLM fine-tunes on these.
  eval       : the other half of test/. BOTH models are scored here and
               neither model ever trains on it.

The test half is split stratified by "category/defect_type", so every defect
type (even rare ones like wood/color with 8 images) appears in both halves.
"""
from __future__ import annotations

import pandas as pd
from sklearn.model_selection import train_test_split

ROLES = ("train_good", "vlm_train", "eval")


def make_split(manifest: pd.DataFrame, seed: int = 42, eval_frac: float = 0.5) -> pd.DataFrame:
    df = manifest.copy().reset_index(drop=True)
    df["stratum"] = df["category"] + "/" + df["defect_type"]
    df["role"] = "train_good"

    test_idx = df.index[df["split"] == "test"]
    vlm_idx, eval_idx = train_test_split(
        test_idx,
        test_size=eval_frac,
        stratify=df.loc[test_idx, "stratum"],
        random_state=seed,
    )
    df.loc[vlm_idx, "role"] = "vlm_train"
    df.loc[eval_idx, "role"] = "eval"
    return df


def check_split(df: pd.DataFrame) -> None:
    """Raise AssertionError if the split leaks or is unbalanced."""
    vlm = set(df.loc[df.role == "vlm_train", "path"])
    ev = set(df.loc[df.role == "eval", "path"])
    tr = set(df.loc[df.role == "train_good", "path"])

    assert not (vlm & ev), f"LEAK: {len(vlm & ev)} images in both vlm_train and eval"
    assert not (tr & ev), "LEAK: a train image is in eval"
    assert df["path"].is_unique, "duplicate paths in manifest"
    assert set(df["role"]) <= set(ROLES), f"unknown role: {set(df['role']) - set(ROLES)}"
    assert (df.loc[df.role == "train_good", "label"] == "good").all(), "train_good must be all good"
    assert (df.loc[df.split == "train", "role"] == "train_good").all(), "train images must stay train_good"

    defect_strata = set(df.loc[df.label == "defect", "stratum"])
    for role in ("vlm_train", "eval"):
        have = set(df.loc[(df.role == role) & (df.label == "defect"), "stratum"])
        missing = defect_strata - have
        assert not missing, f"{role} is missing defect types: {sorted(missing)}"

    for cat, g in df[df.role == "eval"].groupby("category"):
        assert set(g["label"]) == {"good", "defect"}, f"eval/{cat} needs both good and defect images"


## Step 3 - Make the split and run the leakage checks
If any check fails, this cell stops with an `AssertionError` naming the problem.

In [ ]:
df = make_split(manifest, seed=42, eval_frac=0.5)
check_split(df)
print('All leakage checks passed ✓')
df.role.value_counts()

## Step 4 - Summary tables
Expected: train_good 3629, vlm_train 862, eval 863.

In [ ]:
df.pivot_table(index='role', columns='label', values='path', aggfunc='count', fill_value=0)

In [ ]:
# Eval images per category - how much each per-category score can be trusted
ev = df[df.role == 'eval'].pivot_table(index=['kind', 'category'], columns='label',
                                        values='path', aggfunc='count', fill_value=0)
ev['points_per_defect_image'] = (100 / ev['defect']).round(1)
ev

In [ ]:
# The rarest defect types in eval - per-type scores here are noise
(df[(df.role == 'eval') & (df.label == 'defect')]
   .groupby('stratum').size().sort_values().head(10))

## Step 5 - Save
Then click **Save Version → Save & Run All (Commit)** so `splits.csv` becomes a permanent output you can attach as input in Week 2.

In [ ]:
out = Path('/kaggle/working/splits.csv')
df.to_csv(out, index=False)
print('saved', out, len(df), 'rows')

## Findings (write yours)
- Number: eval has ___ images (___ defect, ___ good). The smallest per-category eval defect count is ___ (category ___).
- Meaning:
- Consequence:
